# 01 · PyTorch Lightning 与训练框架  PyTorch Lightning & Training Framework

> **能力标签**：GA4（训练优化 / Training Optimization）

## 目标 Objectives

完成本课后，你应该能够：

1. 描述**训练循环样板**（training-loop boilerplate）的标准结构：前向传播、损失计算、反向传播、优化器步进。
2. 说明 **PyTorch Lightning** 如何把训练循环封装为 `LightningModule`（模块抽象）+ `Trainer`（训练引擎），减少重复代码。
3. 理解 `LightningModule` 的核心钩子（hooks）：`forward`、`training_step`、`configure_optimizers`，以及 `Trainer` 如何调用它们。
4. 用纯 PyTorch 演示与 Lightning 等价的训练逻辑（自包含、可运行）。
5. 描述**回调（callback）**机制：EMA 回调（Exponential Moving Average）的思路，以及它如何与 `w10-ema` 练习对应。

> 对应能力：**GA4**

## 概念讲解 Concepts

### 1. 训练循环样板 Training-Loop Boilerplate

最基础的 PyTorch 训练循环包含以下步骤：

```
for epoch in range(num_epochs):
    for xb, yb in dataloader:
        optimizer.zero_grad()
        y_hat = model(xb)
        loss = loss_fn(y_hat, yb)
        loss.backward()
        optimizer.step()
```

当项目变复杂（多 GPU、混合精度、日志、checkpoint）时，这个循环会膨胀到数百行，且容易出 bug。

---

### 2. PyTorch Lightning 的解决方案

**PyTorch Lightning**（PL）把训练循环的**重复骨架**交给框架管理，用户只需实现**业务逻辑**：

| 用户实现 | PL 框架负责 |
|---------|------------|
| `training_step(batch, batch_idx)` | epoch 循环、设备迁移 |
| `configure_optimizers()` | 反向传播、`optimizer.step()` |
| `forward(x)` | logging、checkpoint、progress bar |

核心类：
- **`LightningModule`**：继承自 `nn.Module`，加入 PL 钩子。
- **`Trainer`**：控制训练流程，接受 `accelerator`、`devices`、`max_epochs` 等参数。
- **`Callback`**：在训练各节点插入自定义逻辑（如 EMA、学习率热身）。

---

### 3. LightningModule 伪代码 Pseudocode

```python
# 伪代码（不可直接运行，需 pytorch_lightning 包）

import pytorch_lightning as pl

class MyModel(pl.LightningModule):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = F.mse_loss(self(x), y)
        self.log("train_loss", loss)   # 自动记录 automatically logged
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

# 启动训练
trainer = pl.Trainer(max_epochs=10, accelerator="cpu")
trainer.fit(model, train_dataloader)
```

`Trainer` 会自动处理：设备管理、进度条、TensorBoard 日志、梯度裁剪等。

---

### 4. Callback 机制 Callback Mechanism

回调（callback）在训练的**生命周期节点**（lifecycle hooks）插入逻辑：

```python
# 伪代码
class EMACallback(pl.Callback):
    def __init__(self, decay=0.99):
        self.decay = decay
        self.ema = None

    def on_train_start(self, trainer, pl_module):
        self.ema = EMA(pl_module, decay=self.decay)

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self.ema.update(pl_module)   # 每步更新影子参数
```

常见回调节点：`on_train_start/end`、`on_train_batch_end`、`on_validation_epoch_end`、`on_save_checkpoint`。

---

### 5. 纯 PyTorch 等价实现 Pure PyTorch Equivalent

Lightning 本质上是**模式化地调用**了同样的纯 PyTorch 操作。本课示例区用纯 PyTorch 演示与 Lightning 完全等价的逻辑，帮助你理解 PL 背后发生了什么。

## 示例 Worked Example

In [ ]:
# Setup — 自包含，CPU，可复现
import tempfile, os
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
DEVICE = torch.device("cpu")
TMPDIR = tempfile.mkdtemp()
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

In [ ]:
# 1. 合成数据集 Synthetic dataset

torch.manual_seed(0)
N = 200
X = torch.randn(N, 2)
y = (X[:, 0] * 1.5 - X[:, 1] * 0.8 + 0.3 + 0.1 * torch.randn(N)).unsqueeze(1)  # (N,1)

def make_batches(X, y, batch_size=32):
    idx = torch.randperm(len(X))
    for s in range(0, len(X), batch_size):
        i = idx[s:s + batch_size]
        yield X[i], y[i]

print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# 2. 模型定义 Model definition

class MLP(nn.Module):
    '''两层 MLP，对应 LightningModule 的 forward。'''
    def __init__(self, in_features=2, hidden=32, out_features=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model = MLP()
print("参数数量 param count:", sum(p.numel() for p in model.parameters()))

In [ ]:
# 3. EMA 类 (mirrors w10-ema)

class EMA:
    '''指数滑动平均 Exponential Moving Average.
    shadow_t = decay * shadow_{t-1} + (1 - decay) * param_t
    mirrors the w10-ema exercise solution.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.99):
        self.decay = decay
        # 深拷贝初始参数作为影子参数 deep-copy initial params as shadow
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        '''每个训练步骤后调用 Call after each training step.'''
        for n, p in model.named_parameters():
            self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        '''将影子参数写回模型（评估时用）Copy shadow params back to model (for eval).'''
        for n, p in model.named_parameters():
            p.copy_(self.shadow[n])

print("EMA 类定义完成 EMA class defined")

In [ ]:
# 4. 训练循环（等价于 Lightning Trainer）Training loop

torch.manual_seed(42)
model = MLP()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ema = EMA(model, decay=0.99)   # 相当于注册 EMACallback

NUM_EPOCHS = 20
history = []

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    model.train()
    for xb, yb in make_batches(X, y, batch_size=32):
        optimizer.zero_grad()          # 清零梯度
        y_hat = model(xb)              # forward  (= LightningModule.forward)
        loss = F.mse_loss(y_hat, yb)   # 损失 loss
        loss.backward()                # 反向传播 backward
        optimizer.step()               # 优化器步进
        ema.update(model)              # 等价于 EMACallback.on_train_batch_end

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    history.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS}  loss={avg_loss:.4f}")

print("\n训练完成 Training done")

In [ ]:
# 5. 使用 EMA 评估 Evaluate with EMA weights

import copy

# 保存当前（live）参数 Save live params
live_state = copy.deepcopy(model.state_dict())

# 切换到 EMA 参数 Switch to EMA params
ema.copy_to(model)
model.eval()

with torch.no_grad():
    pred_ema = model(X)
    loss_ema = F.mse_loss(pred_ema, y).item()

# 恢复 live 参数 Restore live params
model.load_state_dict(live_state)
with torch.no_grad():
    pred_live = model(X)
    loss_live = F.mse_loss(pred_live, y).item()

print(f"Live 模型 MSE  : {loss_live:.4f}")
print(f"EMA  模型 MSE  : {loss_ema:.4f}")

# 快速合理性检验 sanity check
assert loss_live < 1.0, "损失太高，训练可能有问题"
assert len(history) == NUM_EPOCHS
print("\u2713 EMA 评估通过 EMA eval passed")

## 动手 Exercise

在下面的 cell 中，**完成 `train_with_ema`**：

- 接受 `model`、`X`、`y`、`num_epochs`、`decay` 参数
- 实现标准训练循环（Adam, lr=1e-3, batch_size=32）
- 在每个 batch 后调用 `ema.update(model)`
- 返回 `(history, ema)`：`history` 是每个 epoch 的平均 MSE 列表，`ema` 是 `EMA` 对象

In [ ]:
def train_with_ema(model, X, y, num_epochs=10, decay=0.99):
    '''训练循环 + EMA，返回 (history, ema)。
    Training loop + EMA, returns (history, ema).
    '''
    # TODO: 实现
    # Hint: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    #       ema = EMA(model, decay=decay)
    #       for epoch in ...: for xb, yb in make_batches(...):
    #           ... loss.backward() ... optimizer.step() ... ema.update(model)
    raise NotImplementedError("请实现 train_with_ema")

# 实现后取消注释测试 Uncomment after implementation:
# torch.manual_seed(99)
# m_ex = MLP()
# hist_ex, ema_ex = train_with_ema(m_ex, X, y, num_epochs=5)
# print(f"最终损失 final loss: {hist_ex[-1]:.4f}")
# assert len(hist_ex) == 5 and hist_ex[-1] < 0.5
# print("\u2713 train_with_ema 通过 passed")
print("提示：实现 train_with_ema 后取消注释运行验证")

## 延伸阅读 Further Reading

1. **PyTorch Lightning 官方文档**：<https://lightning.ai/docs/pytorch/stable/> — `LightningModule`、`Trainer`、`Callback` API 完整参考。
2. **Lightning 快速入门教程**：<https://lightning.ai/docs/pytorch/stable/starter/introduction.html> — 15 分钟上手指南。
3. **Andrej Karpathy "The spelled-out intro to neural networks and backpropagation"**（YouTube）——深入理解反向传播，是理解训练循环的基础。
4. **PyTorch `torch.nn.Module` 文档**：<https://pytorch.org/docs/stable/generated/torch.nn.Module.html> — `state_dict`、`named_parameters` 等 API。
5. **EMA in Deep Learning**：OpenAI、Stable Diffusion 等项目广泛使用 EMA 稳定评估权重，见 DDPM 代码库（Phil Wang 的 `denoising-diffusion-pytorch`）。

## 关联练习 Related Assignments

```bash
python check.py 01-ema
```

> 实现 `EMA.__init__`（初始化影子参数）、`EMA.update`（EMA 公式）、`EMA.copy_to`（写回模型）。
> 关键：`shadow[n].mul_(decay).add_(p.detach(), alpha=1-decay)`；用 `@torch.no_grad()` 避免构建计算图。

> 能力标签：**GA4** · threshold ≥ 0.7